In [1]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
import torch

torch_dtype = torch.bfloat16
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"

initial_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch_dtype,
    attn_implementation="flash_attention_2",
    device_map={"": 0},
)

from train_gym.rmt.rmt_wrappers import MemoryCell, RecurrentWrapper

cell = MemoryCell(initial_model, num_mem_tokens=16)
model = RecurrentWrapper(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
# tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
# temp = tokenizer("test")
keys = ["input_ids", "attention_mask", "labels"]
test_input_data = {}
for key in keys:
    test_input_data[key] = torch.ones(
        (1, 2048),
        device=initial_model.device,
        dtype=torch.long,
    )
test_input_data

{'input_ids': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'),
 'labels': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')}

In [4]:
model.memory_cell.model.dtype

torch.bfloat16

In [5]:
model.memory_cell.memory.dtype

torch.bfloat16

In [6]:
with torch.no_grad():
    result = model(**test_input_data)

result

CausalLMOutputWithCrossAttentions(loss=tensor(3.5000, device='cuda:0', dtype=torch.bfloat16), logits=tensor([[[ 5.5312,  6.3438,  4.9688,  ..., -1.5703, -1.5703, -1.5703],
         [ 6.2188,  9.5625,  5.4375,  ..., -1.9531, -1.9531, -1.9531],
         [ 7.2500, 10.3750,  6.2812,  ..., -1.8750, -1.8750, -1.8750],
         ...,
         [ 8.3125, 13.5000,  8.5625,  ..., -1.1875, -1.1875, -1.1875],
         [ 5.8750, 10.8125,  5.0000,  ..., -1.5859, -1.5859, -1.5859],
         [ 4.7500,  8.9375,  3.9375,  ..., -2.4219, -2.4219, -2.4219]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None, cross_attentions=None)

In [7]:
result.logits

tensor([[[ 9.1875, 11.6875,  8.4375,  ...,  0.0835,  0.0835,  0.0835],
         [11.6875, 13.1875,  8.8125,  ..., -0.0449, -0.0449, -0.0449],
         [12.1250, 13.1250,  8.3750,  ..., -0.3125, -0.3125, -0.3125],
         ...,
         [ 5.1875, 10.0000,  1.1094,  ..., -2.4062, -2.4062, -2.4062],
         [ 5.1562,  9.9375,  1.0781,  ..., -2.4219, -2.4219, -2.4219],
         [ 5.1250,  9.9375,  1.2109,  ..., -2.3750, -2.3750, -2.3750]]],
       device='cuda:0', dtype=torch.bfloat16)

#### generate

In [ ]:
test_input_data["input_ids"].shape

torch.Size([1, 2048])

In [14]:
result = model.generate(
    # input_ids=test_input_data["input_ids"],
    input_ids=torch.randint(
        low=0,
        high=128,
        # size=(1, 2),
        size=(32, 77),
        dtype=torch.long,
        device="cuda",
    ),
    # attention_mask=test_input_data["attention_mask"],
    max_new_tokens=20,
)
# tokenizer.
result

tensor([[    91,     77,     91,     57,     91,     89,     91,  53498,     61,
             71,     91,     64,     91,     82,     91,     16,     91,     17,
             91,     18],
        [    93,     67,     93,     61,     93,     79,     93,     79,     93,
             67,     93,     61,     93,     84,     93,     59,     93,     93,
             70,     93],
        [    82,      5,     63,     14,     67,      5,     14,     67,     63,
             14,     67,      5,     14,     67,     63,     14,     67,     63,
             14,     67],
        [    45,     59,     61,     77,     59,     63,     62,     82,     13,
             93,     59,     63,     38,     13,     93,      0,     38,     13,
             93,      0],
        [    71,      2,     86,     51,     59,     91,     62,     73,     32,
             59,     91,     62,     73,     32,     59,     91,     62,     73,
             32,     59],
        [    91,     32,     15,     59,     61,     41,    

In [19]:
getattr(model, 'memory_cell',None) is None

False

In [22]:
getattr(model, 'memory_cell',None).memory.shape[0]

16

### Optimize train code

In [2]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
import torch

torch_dtype = torch.bfloat16
model_name_or_path = "unsloth/Llama-3.2-1B-Instruct"

initial_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch_dtype,
    attn_implementation="flash_attention_2",
    device_map={"": 0},
)

In [ ]:
import math
import torch
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import CausalLMOutputWithCrossAttentions


class MemoryCellTrain(torch.nn.Module):
    def __init__(self, base_model, num_mem_tokens):
        super().__init__()
        self.model = base_model
        self.create_memory(num_mem_tokens)

    def create_memory(self, num_mem_tokens):
        self.num_mem_tokens = num_mem_tokens
        embeddings = self.model.get_input_embeddings()
        memory_dim = getattr(self.model.config, "n_embd", self.model.config.hidden_size)
        memory_weights = (
            torch.randn(
                (num_mem_tokens, memory_dim),
                device=self.model.device,
                dtype=self.model.dtype,
            )
            * embeddings.weight.data.std()
        )
        self.register_parameter(
            "memory",
            torch.nn.Parameter(
                memory_weights,
                requires_grad=True,
            ),
        )

        self.read_memory_position = range(num_mem_tokens)
        self.write_memory_position = range(-num_mem_tokens, 0)

    def set_memory(self, input_shape):
        memory = self.memory.repeat(input_shape[0], 1, 1)
        return memory

    def forward(self, input_ids, memory_state=None, **kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        labels = None
        if "labels" in kwargs:
            labels = kwargs.pop("labels")

        seg_kwargs = self.process_input(
            input_ids, memory_state, write_mem=True, **kwargs
        )
        out = self.model(**seg_kwargs)
        kwargs["labels"] = labels
        out, new_memory_state = self.process_output(out, **kwargs)

        return out, new_memory_state

    def generate(self, input_ids, memory_state, attention_mask=None, **generate_kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        seg_kwargs = self.process_input(
            input_ids, memory_state, attention_mask=attention_mask, write_mem=False
        )
        out = self.model.generate(
            inputs_embeds=seg_kwargs["inputs_embeds"],
            attention_mask=seg_kwargs["attention_mask"],
            **generate_kwargs
        )
        return out

    def process_input(self, input_ids, memory_state, write_mem, **kwargs):
        seg_kwargs = dict(**kwargs)

        inputs_embeds = kwargs.get("inputs_embeds")
        if inputs_embeds is None:
            inputs_embeds = self.model.get_input_embeddings()(input_ids)

        if self.num_mem_tokens > 0:
            if write_mem:
                inputs_embeds = torch.cat(
                    [memory_state, inputs_embeds, memory_state], dim=1
                )
            else:
                inputs_embeds = torch.cat([memory_state, inputs_embeds], dim=1)

        seg_kwargs["input_ids"] = None
        seg_kwargs["inputs_embeds"] = inputs_embeds
        if kwargs.get("attention_mask") is not None:
            seg_kwargs["attention_mask"] = self.pad_attention_mask(
                kwargs["attention_mask"], inputs_embeds.shape
            )
        seg_kwargs["output_hidden_states"] = True
        return seg_kwargs

    def pad_attention_mask(self, attention_mask, shape):
        if self.num_mem_tokens in {0, None}:
            return attention_mask
        else:
            mask = torch.ones(*shape[:2], dtype=torch.int64).to(attention_mask.device)
            mask[
                :, self.num_mem_tokens : self.num_mem_tokens + attention_mask.shape[1]
            ] = attention_mask
            return mask

    def process_output(self, model_outputs, **kwargs):
        if self.num_mem_tokens not in {0, None}:
            out = CausalLMOutputWithCrossAttentions()
            memory_state = model_outputs.hidden_states[-1][:, -self.num_mem_tokens :]
            out["logits"] = model_outputs.logits[
                :, self.num_mem_tokens : -self.num_mem_tokens
            ]

            if kwargs.get("output_hidden_states"):
                out["hidden_states"] = [
                    lh[:, self.num_mem_tokens : -self.num_mem_tokens]
                    for lh in model_outputs.hidden_states
                ]
            if kwargs.get("output_attentions"):
                out["attentions"] = model_outputs["attentions"]

            if not kwargs["labels"] is None:
                # loss = self.model.loss_function(
                #     logits=out["logits"],
                #     labels=,
                #     vocab_size=self.model.config.vocab_size,
                # )
                logits = out["logits"]
                labels = kwargs["labels"]
                ignore_index = -100
                logits = logits.float()
                vocab_size = self.model.config.vocab_size
                labels = torch.nn.functional.pad(
                    labels,
                    (0, 1),
                    value=ignore_index,
                )
                shift_labels = labels[..., 1:].contiguous()

                # Flatten the tokens
                logits = logits.view(-1, vocab_size)
                shift_labels = shift_labels.view(-1)
                # Enable model parallelism
                shift_labels = shift_labels.to(logits.device)

                loss = torch.nn.functional.cross_entropy(
                    logits,
                    shift_labels,
                    ignore_index=ignore_index,
                    reduction="sum",
                )
                out["loss"] = loss

            # clean memory while training
            if self.training:
                out["logits"] = None
                out["attentions"] = None
                out["hidden_states"] = None
        else:
            memory_state = None
            out = model_outputs

        return out, memory_state


class RecurrentWrapperTrain(torch.nn.Module):
    def __init__(self, memory_cell, **rmt_kwargs):
        super().__init__()
        self.memory_cell = memory_cell
        self.rmt_config = rmt_kwargs

    def forward(
        self,
        input_ids,
        labels=None,
        labels_mask=None,
        inputs_embeds=None,
        attention_mask=None,
        output_attentions=None,
        output_hidden_states=None,
        num_items_in_batch=None,
    ):
        memory_state = None
        segmented = self.segment(
            input_ids=input_ids,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )

        cell_outputs = []
        for seg_num, segment in enumerate(segmented):
            cell_out, memory_state = self.memory_cell(
                **segment, memory_state=memory_state, output_hidden_states=True
            )
            cell_outputs.append(cell_out)
            memory_state = self.manage_gradients(memory_state, seg_num)

        out = self.process_outputs(
            cell_outputs,
            labels=labels,
            labels_mask=labels_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            num_items_in_batch=num_items_in_batch,
        )
        return out

    def generate(self, input_ids, attention_mask=None, **generate_kwargs):
        memory_state = None
        segmented = self.segment(input_ids=input_ids, attention_mask=attention_mask)

        for seg_num, segment in enumerate(segmented[:-1]):
            cell_out, memory_state = self.memory_cell(
                **segment, memory_state=memory_state, output_hidden_states=True
            )

        final_segment = segmented[-1]
        out = self.memory_cell.generate(
            **final_segment, memory_state=memory_state, **generate_kwargs
        )

        return out

    def segment(self, **kwargs):
        segments = []
        for k, tensor in kwargs.items():
            if tensor is not None:
                k_segments = self.split_tensor(tensor)
                for s, k_seg in enumerate(k_segments):
                    if s < len(segments):
                        segments[s][k] = k_seg
                    else:
                        segments.append({k: k_seg})

        return segments

    def split_tensor(self, tensor):
        align = self.rmt_config.get("segment_alignment")
        segment_size = self.rmt_config.get("segment_size")
        if align in {"left", None}:
            split_inds = list(range(0, tensor.shape[1], segment_size)) + [
                tensor.shape[1]
            ]
            segments = [
                tensor[:, start:end] for (start, end) in zip(split_inds, split_inds[1:])
            ]
        elif align in {"right", None}:
            split_inds = (list(range(tensor.shape[1], 0, -segment_size)) + [0])[::-1]
            segments = [
                tensor[:, start:end] for (start, end) in zip(split_inds, split_inds[1:])
            ]
        elif align == "center":
            n_seg = math.ceil(tensor.shape[1] / segment_size)
            segments = torch.chunk(tensor, n_seg, dim=1)
        else:
            raise NotImplementedError
        return segments

    def process_outputs(self, cell_outputs, **kwargs):
        out = CausalLMOutputWithCrossAttentions()
        if not self.training:
            full_logits = torch.cat([o.logits for o in cell_outputs], dim=1)
            full_hidden_states = tuple(
                [
                    torch.cat(layer_hs, dim=1)
                    for layer_hs in zip(*[o.hidden_states for o in cell_outputs])
                ]
            )

        labels = kwargs.get("labels")
        if labels is not None:
            losses = [o.loss for o in cell_outputs]
            losses = torch.stack(losses, dim=0).sum(dim=0)

            out["loss"] = losses / kwargs["num_items_in_batch"]
        else:
            out["loss"] = 0

        if not self.training:
            out["logits"] = full_logits

        if not self.training:
            if kwargs.get("output_hidden_states"):
                out["hidden_states"] = full_hidden_states

        return out

    def manage_gradients(self, memory_state, seg_num):
        k2, max_n_segments = self.rmt_config.get("k2"), self.rmt_config.get(
            "max_n_segments"
        )
        if seg_num == 0 or k2 in {-1, None} or seg_num + k2 > max_n_segments:
            return memory_state

        memory_state = memory_state.detach()
        return memory_state

    def gradient_checkpointing_enable(self, *args, **kwargs):
        self.memory_cell.model.gradient_checkpointing_enable(*args, **kwargs)


cell = MemoryCellTrain(initial_model, num_mem_tokens=16)
model = RecurrentWrapperTrain(
    cell,
    segment_size=1024,
    max_n_segments=2,
    vary_n_segments=False,
    k2=-1,
)

keys = [
    "input_ids",
    "attention_mask",
    "labels",
]
test_input_data = {}
for key in keys:
    test_input_data[key] = torch.ones(
        (1, 2048),
        device=initial_model.device,
        dtype=torch.long,
    )

# test_input_data['num_items_in_batch'] = 100000
test_input_data["num_items_in_batch"] = 2048
with torch.no_grad():
    result = model(**test_input_data)

result

CausalLMOutputWithCrossAttentions(loss=tensor(3.7612, device='cuda:0'), logits=None, past_key_values=None, hidden_states=None, attentions=None, cross_attentions=None)

In [ ]:
my_list = [torch.tensor(1), torch.tensor(1)]
torch.stack(my_list, dim=0)  # .sum(dim=0)

tensor([1, 1])

In [4]:
initial_model(**test_input_data)

CausalLMOutputWithPast(loss=tensor(7.9089, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[[ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688],
         [ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688],
         [ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688],
         ...,
         [ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688],
         [ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688],
         [ 4.6875,  4.7812,  5.0312,  ..., -1.9688, -1.9688, -1.9688]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=<transformers.cache_utils.DynamicCache object at 0x7fde2cb3cc50>, hidden_states=None, attentions=None)

In [9]:
initial_model.training

False

In [ ]:
# initial_model.eval()
initial_model.train()
None
initial_model.training

True

In [8]:
model.training

True